# Evaluation Pipeline

Submission evaluation notebook for the final best configuration:
`nomic + chunks_fixed300_overlap0 + reranked_top5`.

This notebook evaluates one inference output file in two ways:
Retrieval quality: whether the retrieved contexts contain the gold answer.
Generation quality: whether the generated response matches the gold answer.

For TA testing, point `GOLD_PATH` to the gold answers file that matches the query set used by the inference pipeline.

In [1]:
import json
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple


In [2]:
PIPELINE_NAME = "nomic + chunks_fixed300_overlap0 + reranked_top5"

# Replace GOLD_PATH with the gold file provided by the TA when needed.
GOLD_PATH = "examples/benchmark_unified.json"

INFERENCE_OUTPUT_PATH = "results/output_results.json"
SUMMARY_OUTPUT_PATH = "results/evaluation_summary.json"

ANSWER_MATCH_THRESHOLD = 0.6

print(f"Pipeline       : {PIPELINE_NAME}")
print(f"Gold file      : {GOLD_PATH}")
print(f"Inference file : {INFERENCE_OUTPUT_PATH}")
print(f"Summary output : {SUMMARY_OUTPUT_PATH}")
print(f"Answer match   : {ANSWER_MATCH_THRESHOLD}")


Pipeline       : nomic + chunks_fixed300_overlap0 + reranked_top5
Gold file      : examples/benchmark_unified.json
Inference file : results/output_results.json
Summary output : results/evaluation_summary.json
Answer match   : 0.6


## Alignment Rules

Predictions are matched to gold entries in this order:
1. By `query_id`
2. By normalized `query` text

This lets the notebook work with both your current benchmark-style files and older example outputs.

In [3]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "").lower()
    text = text.replace("'", " ")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenise(text: str) -> List[str]:
    return normalize_text(text).split()


STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "and", "or", "in", "on", "for", "with", "by", "from",
    "that", "this", "it", "as", "at", "its", "their", "his", "her",
    "into", "than", "then", "which", "what", "who", "when", "where",
    "how", "means", "meaning", "describe", "describing", "term", "word",
    "called", "also", "known", "used", "using", "made", "make",
}


def content_tokens(text: str) -> List[str]:
    return [
        token
        for token in tokenise(text)
        if len(token) > 2 and token not in STOPWORDS
    ]


def answer_match_ratio(gold_answer: str, candidate_text: str) -> float:
    answer_tokens = content_tokens(gold_answer)
    if not answer_tokens:
        return 0.0
    candidate_token_set = set(content_tokens(candidate_text))
    overlap = sum(1 for token in answer_tokens if token in candidate_token_set)
    return overlap / len(answer_tokens)


def exact_match(prediction: str, reference: str) -> float:
    return float(normalize_text(prediction) == normalize_text(reference))


def token_prf(prediction: str, reference: str) -> Tuple[float, float, float]:
    pred_tokens = tokenise(prediction)
    ref_tokens = tokenise(reference)
    if not pred_tokens or not ref_tokens:
        return (1.0, 1.0, 1.0) if pred_tokens == ref_tokens else (0.0, 0.0, 0.0)
    common = sum((Counter(pred_tokens) & Counter(ref_tokens)).values())
    if common == 0:
        return 0.0, 0.0, 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 4), round(precision, 4), round(recall, 4)


def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> List[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_gold(path: str) -> Tuple[Dict[str, dict], Dict[str, dict]]:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Gold file not found: {path_obj}")

    gold_by_id = {}
    gold_by_query = {}

    if path_obj.suffix == ".jsonl":
        records = read_jsonl(path_obj)
        for idx, rec in enumerate(records, start=1):
            query_id = str(rec.get("query_id") or f"q_{idx:03d}")
            query = rec.get("question") or rec.get("query") or ""
            answer = rec.get("gold_answer") or rec.get("answer") or ""
            source = rec.get("source") or rec.get("source_file") or "unknown"
            entry = {
                "query_id": query_id,
                "query": query,
                "gold_answer": answer,
                "source": source,
            }
            gold_by_id[query_id] = entry
            gold_by_query[normalize_text(query)] = entry
        return gold_by_id, gold_by_query

    data = read_json(path_obj)

    if isinstance(data, dict) and "sources" in data:
        idx = 1
        for source_block in data["sources"]:
            source = source_block.get("source", "unknown")
            for item in source_block.get("questions", []):
                query_id = str(item.get("query_id") or f"q_{idx:03d}")
                query = item.get("question") or item.get("query") or ""
                answer = item.get("gold_answer") or item.get("answer") or ""
                entry = {
                    "query_id": query_id,
                    "query": query,
                    "gold_answer": answer,
                    "source": source,
                }
                gold_by_id[query_id] = entry
                gold_by_query[normalize_text(query)] = entry
                idx += 1
        return gold_by_id, gold_by_query

    if isinstance(data, dict) and "queries" in data:
        for idx, item in enumerate(data["queries"], start=1):
            query_id = str(item.get("query_id") or f"q_{idx:03d}")
            query = item.get("query") or item.get("question") or ""
            answer = item.get("gold_answer") or item.get("answer") or ""
            entry = {
                "query_id": query_id,
                "query": query,
                "gold_answer": answer,
                "source": item.get("source", "unknown"),
            }
            gold_by_id[query_id] = entry
            gold_by_query[normalize_text(query)] = entry
        return gold_by_id, gold_by_query

    raise ValueError(f"Unsupported gold file format: {path_obj}")


def load_predictions(path: str) -> List[dict]:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Inference output not found: {path_obj}")
    data = read_json(path_obj)
    return data.get("results", [])


def resolve_gold(prediction: dict, gold_by_id: Dict[str, dict], gold_by_query: Dict[str, dict]) -> Tuple[Optional[dict], Optional[str]]:
    query_id = str(prediction.get("query_id", "")).strip()
    query = prediction.get("query", "")
    if query_id and query_id in gold_by_id:
        return gold_by_id[query_id], "query_id"
    query_key = normalize_text(query)
    if query_key and query_key in gold_by_query:
        return gold_by_query[query_key], "query"
    return None, None


In [4]:
def evaluate_retrieval(retrieved_context: List[dict], gold_answer: str) -> dict:
    match_rank = None
    best_match_ratio = 0.0

    for rank, item in enumerate(retrieved_context, start=1):
        match_ratio = answer_match_ratio(gold_answer, item.get("text", ""))
        best_match_ratio = max(best_match_ratio, match_ratio)
        if match_rank is None and match_ratio >= ANSWER_MATCH_THRESHOLD:
            match_rank = rank

    return {
        "best_match_ratio": round(best_match_ratio, 4),
        "match_rank": match_rank,
        "hit_at_1": float(match_rank == 1),
        "hit_at_5": float(match_rank is not None and match_rank <= 5),
        "mrr": round(1.0 / match_rank, 4) if match_rank else 0.0,
        "retrieved_count": len(retrieved_context),
    }


def evaluate_generation(response: str, gold_answer: str) -> dict:
    f1, precision, recall = token_prf(response, gold_answer)
    return {
        "exact_match": exact_match(response, gold_answer),
        "token_f1": f1,
        "token_precision": precision,
        "token_recall": recall,
        "answer_match_ratio": round(answer_match_ratio(gold_answer, response), 4),
        "response_words": len(response.split()),
        "reference_words": len(gold_answer.split()),
    }


def average(entries: List[dict], key: str) -> float:
    values = [entry[key] for entry in entries if key in entry]
    return round(sum(values) / len(values), 4) if values else 0.0


In [5]:
gold_by_id, gold_by_query = load_gold(GOLD_PATH)
predictions = load_predictions(INFERENCE_OUTPUT_PATH)

per_query = []
missing_predictions = []

for prediction in predictions:
    gold, matched_by = resolve_gold(prediction, gold_by_id, gold_by_query)
    if not gold:
        missing_predictions.append({
            "query_id": prediction.get("query_id"),
            "query": prediction.get("query"),
        })
        continue

    retrieval_scores = evaluate_retrieval(prediction.get("retrieved_context", []), gold["gold_answer"])
    generation_scores = evaluate_generation(prediction.get("response", "").strip(), gold["gold_answer"])

    per_query.append({
        "query_id": gold["query_id"],
        "query": gold["query"],
        "matched_by": matched_by,
        "source": gold.get("source", "unknown"),
        "gold_answer": gold["gold_answer"],
        "response": prediction.get("response", "").strip(),
        "retrieved_context": prediction.get("retrieved_context", []),
        **retrieval_scores,
        **generation_scores,
    })

summary = {
    "pipeline": PIPELINE_NAME,
    "gold_path": GOLD_PATH,
    "inference_output_path": INFERENCE_OUTPUT_PATH,
    "answer_match_threshold": ANSWER_MATCH_THRESHOLD,
    "n_predictions": len(predictions),
    "n_matched": len(per_query),
    "n_missing": len(missing_predictions),
    "retrieval": {
        "hit_at_1": average(per_query, "hit_at_1"),
        "hit_at_5": average(per_query, "hit_at_5"),
        "mrr": average(per_query, "mrr"),
        "best_match_ratio": average(per_query, "best_match_ratio"),
        "avg_retrieved_count": average(per_query, "retrieved_count"),
    },
    "generation": {
        "exact_match": average(per_query, "exact_match"),
        "token_f1": average(per_query, "token_f1"),
        "token_precision": average(per_query, "token_precision"),
        "token_recall": average(per_query, "token_recall"),
        "answer_match_ratio": average(per_query, "answer_match_ratio"),
        "avg_response_words": average(per_query, "response_words"),
    },
    "missing_predictions": missing_predictions,
    "per_query": per_query,
}

output_path = Path(SUMMARY_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("=" * 80)
print("Submission Evaluation Summary")
print("=" * 80)
print(f"Pipeline            : {PIPELINE_NAME}")
print(f"Predictions loaded  : {len(predictions)}")
print(f"Matched to gold     : {len(per_query)}")
print(f"Missing gold match  : {len(missing_predictions)}")
print()
print("Retrieval metrics")
print(f"  Hit@1            : {summary['retrieval']['hit_at_1']:.4f}")
print(f"  Hit@5            : {summary['retrieval']['hit_at_5']:.4f}")
print(f"  MRR              : {summary['retrieval']['mrr']:.4f}")
print(f"  Best match ratio : {summary['retrieval']['best_match_ratio']:.4f}")
print()
print("Generation metrics")
print(f"  Exact Match      : {summary['generation']['exact_match']:.4f}")
print(f"  Token F1         : {summary['generation']['token_f1']:.4f}")
print(f"  Token Precision  : {summary['generation']['token_precision']:.4f}")
print(f"  Token Recall     : {summary['generation']['token_recall']:.4f}")
print(f"  Answer overlap   : {summary['generation']['answer_match_ratio']:.4f}")
print()
print(f"Saved summary to: {output_path}")


Submission Evaluation Summary
Pipeline            : nomic + chunks_fixed300_overlap0 + reranked_top5
Predictions loaded  : 72
Matched to gold     : 72
Missing gold match  : 0

Retrieval metrics
  Hit@1            : 0.8611
  Hit@5            : 0.9444
  MRR              : 0.9005
  Best match ratio : 0.8677

Generation metrics
  Exact Match      : 0.0000
  Token F1         : 0.2807
  Token Precision  : 0.1894
  Token Recall     : 0.7133
  Answer overlap   : 0.6793

Saved summary to: results\evaluation_summary.json


## Usage

1. Run the submission inference pipeline first to produce `results/output_results.json`.
2. Point `GOLD_PATH` to the matching gold answers file.
3. Run all cells in this notebook.
4. Read the summary in `results/evaluation_summary.json`.
